In [4]:
# ─────────────────────────────────────────────────────────────
# Etapa 1 — Exploração da API da Câmara dos Deputados
# Objetivo: entender estrutura, campos e limitações da API
# ─────────────────────────────────────────────────────────────

import requests
import json
import pandas as pd
from pprint import pprint

# URL base da API — nunca muda
BASE_URL = "https://dadosabertos.camara.leg.br/api/v2"

print("✅ Bibliotecas importadas com sucesso!")
print(f"📡 API base: {BASE_URL}")

✅ Bibliotecas importadas com sucesso!
📡 API base: https://dadosabertos.camara.leg.br/api/v2


In [5]:
# ─────────────────────────────────────────────────────────────
# Bloco 1 — Explorando o endpoint /deputados
# ─────────────────────────────────────────────────────────────

# Fazendo a primeira chamada — apenas 5 registros para explorar
resp = requests.get(f"{BASE_URL}/deputados", params={"itens": 5})

# Verificando se a chamada foi bem sucedida
print(f"Status HTTP: {resp.status_code}")  # 200 = sucesso

# Inspecionando a estrutura raiz da resposta
data = resp.json()
print(f"\nChaves da resposta: {list(data.keys())}")
print(f"Quantidade de deputados retornados: {len(data['dados'])}")

Status HTTP: 200

Chaves da resposta: ['dados', 'links']
Quantidade de deputados retornados: 5


In [6]:
# ─────────────────────────────────────────────────────────────
# Visualizando o primeiro deputado completo
# ─────────────────────────────────────────────────────────────

primeiro_deputado = data["dados"][0]

print("=== ESTRUTURA DE UM DEPUTADO ===")
pprint(primeiro_deputado)

print(f"\n=== CAMPOS DISPONÍVEIS ===")
for campo, valor in primeiro_deputado.items():
    print(f"  {campo}: {valor}")

=== ESTRUTURA DE UM DEPUTADO ===
{'email': 'dep.acaciofavacho@camara.leg.br',
 'id': 204379,
 'idLegislatura': 57,
 'nome': 'Acácio Favacho',
 'siglaPartido': 'MDB',
 'siglaUf': 'AP',
 'uri': 'https://dadosabertos.camara.leg.br/api/v2/deputados/204379',
 'uriPartido': 'https://dadosabertos.camara.leg.br/api/v2/partidos/36899',
 'urlFoto': 'https://www.camara.leg.br/internet/deputado/bandep/204379.jpg'}

=== CAMPOS DISPONÍVEIS ===
  id: 204379
  uri: https://dadosabertos.camara.leg.br/api/v2/deputados/204379
  nome: Acácio Favacho
  siglaPartido: MDB
  uriPartido: https://dadosabertos.camara.leg.br/api/v2/partidos/36899
  siglaUf: AP
  idLegislatura: 57
  urlFoto: https://www.camara.leg.br/internet/deputado/bandep/204379.jpg
  email: dep.acaciofavacho@camara.leg.br


In [7]:
# ─────────────────────────────────────────────────────────────
# Entendendo a paginação — quantas páginas existem?
# ─────────────────────────────────────────────────────────────

print("=== ESTRUTURA DE PAGINAÇÃO (links) ===")
for link in data["links"]:
    print(f"  {link['rel']}: {link['href']}")

# Extraindo o total de páginas do link 'last'
for link in data["links"]:
    if link["rel"] == "last":
        href = link["href"]
        # Pegando o número da última página
        pagina_final = href.split("pagina=")[1].split("&")[0]
        print(f"\n📄 Total de páginas com 5 itens por página: {pagina_final}")
        print(f"📊 Estimativa de deputados: ~{int(pagina_final) * 5}")

=== ESTRUTURA DE PAGINAÇÃO (links) ===
  self: https://dadosabertos.camara.leg.br/api/v2/deputados?itens=5
  next: https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=2&itens=5
  first: https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=1&itens=5
  last: https://dadosabertos.camara.leg.br/api/v2/deputados?pagina=103&itens=5

📄 Total de páginas com 5 itens por página: 103
📊 Estimativa de deputados: ~515


In [8]:
# ─────────────────────────────────────────────────────────────
# Bloco 2 — Explorando o endpoint /proposicoes
# ⚠️ Esse endpoint EXIGE filtro de data — sem ele retorna erro
# ─────────────────────────────────────────────────────────────

resp_prop = requests.get(f"{BASE_URL}/proposicoes", params={
    "dataInicio": "2024-11-01",
    "dataFim":    "2024-11-15",
    "itens":      5
})

print(f"Status HTTP: {resp_prop.status_code}")

data_prop = resp_prop.json()
print(f"Campos na resposta: {list(data_prop.keys())}")
print(f"Proposições retornadas: {len(data_prop['dados'])}")

print("\n=== PRIMEIRA PROPOSIÇÃO ===")
pprint(data_prop["dados"][0])

Status HTTP: 200
Campos na resposta: ['dados', 'links']
Proposições retornadas: 5

=== PRIMEIRA PROPOSIÇÃO ===
{'ano': 2003,
 'codTipo': 139,
 'dataApresentacao': '2003-02-18T15:14',
 'ementa': 'Altera o Decreto-Lei nº 73, de 21 de novembro de 1966, fixando '
           'prazo máximo para pagamento de indenização de sinistros por parte '
           'das sociedades seguradoras e estabelecendo a multa aplicável no '
           'caso de seu descumprimento e dá outras providências.',
 'id': 104417,
 'numero': 46,
 'siglaTipo': 'PL',
 'uri': 'https://dadosabertos.camara.leg.br/api/v2/proposicoes/104417'}


In [9]:
# ─────────────────────────────────────────────────────────────
# Buscando campos extras de uma proposição específica
# ─────────────────────────────────────────────────────────────

prop_id = data_prop["dados"][0]["id"]

resp_detalhe = requests.get(f"{BASE_URL}/proposicoes/{prop_id}")
detalhe = resp_detalhe.json()["dados"]

print("=== CAMPOS EXTRAS NO DETALHE ===")
print(f"Ementa:            {detalhe.get('ementa', 'N/A')}")
print(f"Ementa detalhada:  {detalhe.get('ementaDetalhada', 'N/A')}")
print(f"Palavras-chave:    {detalhe.get('keywords', 'N/A')}")
print(f"Situação:          {detalhe.get('statusProposicao', {}).get('descricaoSituacao', 'N/A')}")
print(f"Regime:            {detalhe.get('statusProposicao', {}).get('regime', 'N/A')}")
print(f"Data apresentação: {detalhe.get('dataApresentacao', 'N/A')}")

print("\n=== TODOS OS CAMPOS DISPONÍVEIS ===")
for campo, valor in detalhe.items():
    print(f"  {campo}: {type(valor).__name__} = {str(valor)[:80]}")

=== CAMPOS EXTRAS NO DETALHE ===
Ementa:            Altera o Decreto-Lei nº 73, de 21 de novembro de 1966, fixando prazo máximo para pagamento de indenização de sinistros por parte das sociedades seguradoras e estabelecendo a multa aplicável no caso de seu descumprimento e dá outras providências.
Ementa detalhada:  
Palavras-chave:    Alteração, Lei do Seguro Privado, obrigatoriedade, sociedade seguradora, inclusão, cláusula, contrato, seguros, fixação, prazo, pagamento, indenização, segurado, sinistro, seguro obrigatório, tempo, apuração, valor, multa,  infrator.
Situação:          Aguardando Designação de Relator(a)
Regime:            Ordinário (Art. 151, III, RICD)
Data apresentação: 2003-02-18T15:14

=== TODOS OS CAMPOS DISPONÍVEIS ===
  id: int = 104417
  uri: str = https://dadosabertos.camara.leg.br/api/v2/proposicoes/104417
  siglaTipo: str = PL
  codTipo: int = 139
  numero: int = 46
  ano: int = 2003
  ementa: str = Altera o Decreto-Lei nº 73, de 21 de novembro de 1966, fixand

In [10]:
# ─────────────────────────────────────────────────────────────
# Bloco 3 — Explorando o endpoint /votacoes
# ─────────────────────────────────────────────────────────────

resp_vot = requests.get(f"{BASE_URL}/votacoes", params={
    "dataInicio": "2024-11-01",
    "dataFim":    "2024-11-15",
    "itens":      5
})

print(f"Status HTTP: {resp_vot.status_code}")

data_vot = resp_vot.json()
print(f"Votações retornadas: {len(data_vot['dados'])}")

print("\n=== PRIMEIRA VOTAÇÃO ===")
pprint(data_vot["dados"][0])

print("\n=== CAMPOS DISPONÍVEIS ===")
for campo, valor in data_vot["dados"][0].items():
    print(f"  {campo}: {str(valor)[:80]}")

Status HTTP: 200
Votações retornadas: 5

=== PRIMEIRA VOTAÇÃO ===
{'aprovacao': 1,
 'data': '2024-11-12',
 'dataHoraRegistro': '2024-11-26T19:55:28',
 'descricao': 'Aprovado o Parecer com Complementação de Voto.',
 'id': '2445282-28',
 'proposicaoObjeto': None,
 'siglaOrgao': 'CPD',
 'uri': 'https://dadosabertos.camara.leg.br/api/v2/votacoes/2445282-28',
 'uriEvento': 'https://dadosabertos.camara.leg.br/api/v2/eventos/74614',
 'uriOrgao': 'https://dadosabertos.camara.leg.br/api/v2/orgaos/537480',
 'uriProposicaoObjeto': None}

=== CAMPOS DISPONÍVEIS ===
  id: 2445282-28
  uri: https://dadosabertos.camara.leg.br/api/v2/votacoes/2445282-28
  data: 2024-11-12
  dataHoraRegistro: 2024-11-26T19:55:28
  siglaOrgao: CPD
  uriOrgao: https://dadosabertos.camara.leg.br/api/v2/orgaos/537480
  uriEvento: https://dadosabertos.camara.leg.br/api/v2/eventos/74614
  proposicaoObjeto: None
  uriProposicaoObjeto: None
  descricao: Aprovado o Parecer com Complementação de Voto.
  aprovacao: 1


In [11]:
# ─────────────────────────────────────────────────────────────
# Bloco 4 — Explorando /partidos e /despesas
# ─────────────────────────────────────────────────────────────

# --- Partidos ---
resp_part = requests.get(f"{BASE_URL}/partidos", params={"itens": 5})
data_part = resp_part.json()

print(f"Status partidos: {resp_part.status_code}")
print(f"Partidos retornados: {len(data_part['dados'])}")
print("\n=== PRIMEIRO PARTIDO ===")
pprint(data_part["dados"][0])

# --- Despesas de um deputado ---
dep_id = 204379  # Acácio Favacho — ID que já temos

resp_desp = requests.get(f"{BASE_URL}/deputados/{dep_id}/despesas", params={
    "ano":   2024,
    "mes":   11,
    "itens": 5
})
data_desp = resp_desp.json()

print(f"\nStatus despesas: {resp_desp.status_code}")
print(f"Despesas retornadas: {len(data_desp['dados'])}")

if data_desp["dados"]:
    print("\n=== PRIMEIRA DESPESA ===")
    pprint(data_desp["dados"][0])
else:
    print("⚠️ Nenhuma despesa encontrada para esse deputado nesse mês")

Status partidos: 200
Partidos retornados: 5

=== PRIMEIRO PARTIDO ===
{'id': 36898,
 'nome': 'Avante',
 'sigla': 'AVANTE',
 'uri': 'https://dadosabertos.camara.leg.br/api/v2/partidos/36898'}

Status despesas: 200
Despesas retornadas: 5

=== PRIMEIRA DESPESA ===
{'ano': 2024,
 'cnpjCpfFornecedor': '08532429000131',
 'codDocumento': '7837655',
 'codLote': 2092841,
 'codTipoDocumento': 0,
 'dataDocumento': '2024-11-06T00:00:00',
 'mes': 11,
 'nomeFornecedor': 'AMORETTO CAFES EXPRESSO LTDA',
 'numDocumento': '1735',
 'numRessarcimento': '',
 'parcela': 0,
 'tipoDespesa': 'MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE PARLAMENTAR',
 'tipoDocumento': 'Nota Fiscal',
 'urlDocumento': 'https://www.camara.leg.br/cota-parlamentar/documentos/publ/3308/2024/7837655.pdf',
 'valorDocumento': 750.0,
 'valorGlosa': 0.0,
 'valorLiquido': 750.0}


In [12]:
# ─────────────────────────────────────────────────────────────
# Bloco 5 — Análise rápida com Pandas
# ─────────────────────────────────────────────────────────────

# Carregando 100 deputados para análise
resp_df = requests.get(f"{BASE_URL}/deputados", params={"itens": 100})
df_dep = pd.DataFrame(resp_df.json()["dados"])

print(f"Shape: {df_dep.shape}")
print(f"\nColunas: {list(df_dep.columns)}")

print(f"\n=== VALORES NULOS ===")
print(df_dep.isnull().sum())

print(f"\n=== TOP 10 PARTIDOS ===")
print(df_dep["siglaPartido"].value_counts().head(10))

print(f"\n=== DEPUTADOS POR UF ===")
print(df_dep["siglaUf"].value_counts())

Shape: (100, 9)

Colunas: ['id', 'uri', 'nome', 'siglaPartido', 'uriPartido', 'siglaUf', 'idLegislatura', 'urlFoto', 'email']

=== VALORES NULOS ===
id               0
uri              0
nome             0
siglaPartido     0
uriPartido       0
siglaUf          0
idLegislatura    0
urlFoto          0
email            0
dtype: int64

=== TOP 10 PARTIDOS ===
siglaPartido
PL              18
PP              15
PT              14
MDB             12
REPUBLICANOS     8
PSD              6
UNIÃO            5
PV               4
PSDB             4
PDT              3
Name: count, dtype: int64

=== DEPUTADOS POR UF ===
siglaUf
SP    16
BA     9
RS     8
RJ     8
MG     6
PE     5
PR     4
SC     4
PA     4
CE     4
MA     4
AM     4
AP     3
TO     3
AL     2
DF     2
PB     2
GO     2
PI     2
RN     2
MS     2
RR     1
ES     1
AC     1
MT     1
Name: count, dtype: int64


In [ ]:
# ─────────────────────────────────────────────────────────────
# Bloco 6 — Decisões de campo e hipóteses do projeto
# ─────────────────────────────────────────────────────────────

decisoes = """
╔══════════════════════════════════════════════════════════════╗
║           DECISÕES DE MODELAGEM — BÚSSOLA PÚBLICA           ║
╚══════════════════════════════════════════════════════════════╝

TABELA: deputados (dimensão)
  ✅ USAR:    id, nome, siglaPartido, siglaUf, idLegislatura, email, urlFoto
  ❌ IGNORAR: uri, uriPartido (URLs internas desnecessárias)

TABELA: partidos (dimensão)
  ✅ USAR:    id, sigla, nome
  ❌ IGNORAR: uri

TABELA: proposicoes (fato)
  ✅ USAR:    id, siglaTipo, numero, ano, ementa, keywords,
              dataApresentacao, descricaoTipo
              statusProposicao.descricaoSituacao (campo aninhado!)
              statusProposicao.regime (campo aninhado!)
  ❌ IGNORAR: uri, uriAutores, uriPropPrincipal (URLs internas)
  ⚠️  ATENÇÃO: statusProposicao é dict — usar json_normalize() na Etapa 3

TABELA: votacoes (fato)
  ✅ USAR:    id, data, dataHoraRegistro, descricao, aprovacao, siglaOrgao
  ❌ IGNORAR: uri, uriOrgao, uriEvento
  ⚠️  ATENÇÃO: proposicaoObjeto pode ser None — tratar como nullable

TABELA: despesas (fato)
  ✅ USAR:    ano, mes, tipoDespesa, nomeFornecedor,
              valorDocumento, valorLiquido, dataDocumento
  ❌ IGNORAR: cnpjCpfFornecedor (dado sensível)
  ⚠️  ATENÇÃO: valorLiquido pode ser negativo (estorno) — tratar na validação
"""

hipoteses = """
╔══════════════════════════════════════════════════════════════╗
║         HIPÓTESES QUE O PIPELINE VAI RESPONDER              ║
╚══════════════════════════════════════════════════════════════╝

1. Quais temas dominam as proposições desta semana?
   → proposicoes + classificação por IA (Etapa 4)

2. Quais deputados são mais ativos (mais proposições)?
   → proposicoes agrupado por autor

3. Como cada partido vota em pautas de tecnologia?
   → votacoes + votos individuais + join deputados

4. Quais deputados gastam mais com a cota parlamentar?
   → despesas agregado por deputado

5. Quais proposições de tecnologia tramitam há mais tempo?
   → proposicoes filtrado por tema + dataApresentacao

6. Algum projeto de lei tributário foi aprovado essa semana?
   → proposicoes + votacoes + classificação IA = Tributário
"""

print(decisoes)
print(hipoteses)


╔══════════════════════════════════════════════════════════════╗
║           DECISÕES DE MODELAGEM — BÚSSOLA PÚBLICA           ║
╚══════════════════════════════════════════════════════════════╝

TABELA: deputados (dimensão)
  ✅ USAR:    id, nome, siglaPartido, siglaUf, idLegislatura, email, urlFoto
  ❌ IGNORAR: uri, uriPartido (URLs internas desnecessárias)

TABELA: partidos (dimensão)
  ✅ USAR:    id, sigla, nome
  ❌ IGNORAR: uri

TABELA: proposicoes (fato)
  ✅ USAR:    id, siglaTipo, numero, ano, ementa, keywords,
              dataApresentacao, descricaoTipo
              statusProposicao.descricaoSituacao (campo aninhado!)
              statusProposicao.regime (campo aninhado!)
  ❌ IGNORAR: uri, uriAutores, uriPropPrincipal (URLs internas)
  ⚠️  ATENÇÃO: statusProposicao é dict — usar json_normalize() na Etapa 3

TABELA: votacoes (fato)
  ✅ USAR:    id, data, dataHoraRegistro, descricao, aprovacao, siglaOrgao
  ❌ IGNORAR: uri, uriOrgao, uriEvento
  ⚠️  ATENÇÃO: proposicaoObjeto po

: 